# Задание 2. Визуализация траекторий

Ноутбук использует те же параметры, что и `main.py`, и строит те же графики для отчёта.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from main import (
    BOX,
    SADDLE,
    GLOBAL_MIN,
    START,
    gradient_descent,
    heavy_ball,
    torch_sgd_momentum,
    make_grid,
)

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


In [ ]:
gd_result = gradient_descent(
    start=START,
    lr=0.7,
    max_iters=200,
    grad_tol=5e-6,
    func_tol=1e-9,
)
fixed_steps = gd_result.iterations

gd_fixed = gradient_descent(start=START, lr=0.7, max_iters=fixed_steps)
rms_escape = heavy_ball(start=START, lr=0.8, gamma=0.99, max_iters=fixed_steps)
rms_zigzag = heavy_ball(start=START, lr=1.3, gamma=0.97, max_iters=fixed_steps)
rms_direct = heavy_ball(start=START, lr=0.5, gamma=0.99, max_iters=fixed_steps)

torch_result = None
if TORCH_AVAILABLE:
    torch_result = torch_sgd_momentum(START, lr=0.8, gamma=0.99, max_iters=fixed_steps)

print(f"Количество итераций: {fixed_steps}")
print(f"Начальная точка: {START}")
print(f"Конечная точка GD: {gd_fixed.final_point}")
print(f"Конечная точка RMSProp: {rms_escape.final_point}")
if torch_result is not None:
    print(f"Конечная точка PyTorch RMSProp: {torch_result.final_point}")


In [ ]:
def plot_contours_inline(xlim, ylim, trajectories, title):
    xx, yy, zz = make_grid(xlim, ylim)
    plt.figure(figsize=(9, 7))
    contours = plt.contour(xx, yy, zz, levels=24, cmap="viridis")
    plt.clabel(contours, inline=True, fontsize=8)

    xmin, xmax, ymin, ymax = BOX
    plt.plot(
        [xmin, xmax, xmax, xmin, xmin],
        [ymin, ymin, ymax, ymax, ymin],
        color="black",
        linewidth=1.2,
        linestyle="--",
        label="граница области",
        zorder=2,
    )
    plt.scatter(*SADDLE, color="red", s=90, marker="x", label="седловая точка", zorder=5)
    plt.scatter(*GLOBAL_MIN, color="green", s=180, marker="*", edgecolors="black", linewidths=0.7, label="глобальный минимум", zorder=6)
    plt.scatter(*START, color="orange", s=120, marker="o", edgecolors="black", linewidths=0.7, label="начальное приближение", zorder=6)
    plt.annotate("x0", xy=START, xytext=(8, 8), textcoords="offset points", fontsize=10, color="black")
    plt.annotate("xmin", xy=GLOBAL_MIN, xytext=(8, 10), textcoords="offset points", fontsize=10, color="black")

    for label, history, color in trajectories:
        plt.plot(history[:, 0], history[:, 1], color=color, linewidth=2, label=label)
        step = max(1, len(history) // 20)
        plt.scatter(history[::step, 0], history[::step, 1], color=color, s=16)

    x_pad = 0.08 * (xlim[1] - xlim[0])
    y_pad = 0.08 * (ylim[1] - ylim[0])
    plt.xlim(xlim[0] - x_pad, xlim[1] + x_pad)
    plt.ylim(ylim[0] - y_pad, ylim[1] + y_pad)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
contour_xlim = (-4.0, 1.0)
contour_ylim = (-4.0, 3.0)

plot_contours_inline(
    contour_xlim,
    contour_ylim,
    [("градиентный спуск", gd_fixed.history, "#d94841")],
    "Задание 2.1: градиентный спуск приближается к седловой точке",
)

plot_contours_inline(
    contour_xlim,
    contour_ylim,
    [
        ("градиентный спуск", gd_fixed.history, "#d94841"),
        ("RMSProp", rms_escape.history, "#2b8cbe"),
    ],
    "Задание 2.2: RMSProp преодолевает окрестность седловой точки",
)

plot_contours_inline(
    contour_xlim,
    contour_ylim,
    [
        ("пилообразное движение", rms_zigzag.history, "#f16913"),
        ("более направленное движение", rms_direct.history, "#238b45"),
    ],
    "Задание 2.3: одна область, разные гиперпараметры RMSProp",
)

if torch_result is not None:
    plot_contours_inline(
        contour_xlim,
        contour_ylim,
        [("PyTorch RMSProp", torch_result.history, "#6a51a3")],
        "Задание 2.4: готовая реализация torch.optim.RMSprop",
    )
